# BioEmu2 C-GAIMMD on trypsin with benzamidine

This tutorial runs BioEmu2 C-GAIMMD on the unbinding of benzamidinium from
bovine trypsin, in explicit water. It builds on tutorial 5 and shows what a
ligand changes:

1. the BioEmu2 system holds the protein and the ligand as a SMILES string; the
   ligand's atoms are mapped to the MD atoms by matching their chemical graphs
   (RDKit), and the folder records BioEmu2's ligand atom order;
2. the completion from the template adds the hydrogens and the Ca²⁺ ion, which
   BioEmu2 does not model;
3. a system-specific validity check: the Ca²⁺ binding site must be intact;
4. the network reads pooled features of the ligand, the pocket and the whole
   protein, and adapts the pair embedding as well as the single one;
5. the fit weights the ligand atoms more in the denoising loss and trains the
   generator from the first round.

The states are defined on the distance between Asp189 CG and the
benzamidine carbon C: bound A below 4.1 Å, unbound B above 15.8 Å.

> **Outputs pending.** This tutorial needs a GPU and GROMACS, the `bioemu2-cuda` extra and the BioEmu2 weights in the Hugging Face cache. The page
> shows the code and the explanations; the outputs of a run on a GPU
> workstation will be added. Every path that depends on your machine is set in
> the first code cell.

## 0. Paths and settings

The system folder holds `run.gro` and `topol.top` (Amber ff14SB with GAFF2 for
the ligand, TIP3P), with the solute in this order: protein, benzamidinium
(`BEN`), the Ca²⁺ ion; then water and ions. It also holds `run.mdp`, `em.mdp`
and `relax.mdp` (without `-DPOSRES`), `states.py`, unbiased MD of each state,
and a reference structure of the bound complex for the canonical pose of the
features (for example built from a crystal structure).

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import aimmd
import MDAnalysis as mda
import numpy as np

# --- configure ---------------------------------------------------------------------
SYSTEM = Path("/path/to/trypsin_benzamidine_system")   # run.gro topol.top *.itp *.mdp states.py
FREE_MD = {"A": SYSTEM / "free_A.xtc", "B": SYSTEM / "free_B.xtc"}   # unbiased MD of each state
REFERENCE = SYSTEM / "reference_bound.gro"              # the bound pose of the same MD system
LIGAND = "BEN"                                         # residue name of the ligand
LIGAND_SMILES = "NC(=[NH2+])c1ccccc1"                  # benzamidinium
ION = "CA"                                             # residue name of the structural Ca2+ ion
CAMPAIGN = Path("trypsin_cgaimmd").resolve()
RUN = Path("/local/scratch/trypsin_cgaimmd_run")
GMX = shutil.which("gmx") or "/path/to/gromacs/bin/gmx"
THREADS = 12
os.environ.setdefault("HF_HUB_OFFLINE", "1")
# ------------------------------------------------------------------------------------

CAMPAIGN.mkdir(parents=True, exist_ok=True)
for item in SYSTEM.iterdir():
    if item.suffix in (".gro", ".top", ".itp", ".mdp", ".py") or item.name.endswith(".ff"):
        target = CAMPAIGN / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy(item, target)
os.chdir(CAMPAIGN)
sys.path.insert(0, str(CAMPAIGN))
from states import states_function                     # noqa: E402

## 1. The solute, its parts and the polish files

The generator atoms are the solute: protein, ligand and ion. The topology
check runs on protein and ligand (`N_PROTEIN_LIGAND` atoms): the ion's
coordination bonds at 2.4 Å would read as covalent bonds under the
covalent-radius rule, so the ion gets its own check below.

In [ ]:
u = mda.Universe("run.gro")
protein = u.select_atoms("protein")
ligand = u.select_atoms(f"resname {LIGAND}")
ion = u.select_atoms(f"resname {ION}")
N_PROTEIN_LIGAND = protein.n_atoms + ligand.n_atoms
N_SOLUTE = N_PROTEIN_LIGAND + ion.n_atoms
order = np.concatenate([protein.indices, ligand.indices, ion.indices])
assert (order == np.arange(N_SOLUTE)).all(), "the solute must come first: protein, ligand, ion"
u.atoms[:N_SOLUTE].write("solute.gro")
u.atoms[:N_PROTEIN_LIGAND].write("protein_ligand.gro")


def molecules_topology(source, target, keep):
    """A copy of `source` whose [ molecules ] keeps only the molecules named in `keep`."""
    lines, section = [], None
    for line in Path(source).read_text().splitlines():
        stripped = line.split(";")[0].strip()
        if stripped.startswith("["):
            section = stripped.strip("[] ").lower()
        elif section == "molecules" and stripped and stripped.split()[0] not in keep:
            continue
        lines.append(line)
    Path(target).write_text("\n".join(lines) + "\n")


# the molecule names of [ molecules ] in topol.top: check them for your topology
molecules_topology("topol.top", "topol_solute.top", keep=("Protein_chain_A", LIGAND, ION))
molecules_topology("topol.top", "topol_protein_ligand.top", keep=("Protein_chain_A", LIGAND))

with mda.Writer("clash_reference_protein_ligand.xtc", N_PROTEIN_LIGAND) as writer:
    for state in "AB":
        md_u = mda.Universe("run.gro", str(FREE_MD[state]))
        for ts in md_u.trajectory[::50]:
            writer.write(md_u.atoms[:N_PROTEIN_LIGAND])
print(N_SOLUTE, "solute atoms:", protein.n_atoms, "protein,", ligand.n_atoms, "ligand,",
      ion.n_atoms, "ion")

## 2. The BioEmu2 system folder

The BioEmu2 system lists the protein chain and the ligand as a SMILES string.
`build-system` maps the protein residue by residue and the ligand by an
isomorphism of its element-labelled heavy-atom graph (the first one in sorted
order, so that a symmetric ligand such as benzamidinium has a fixed
convention). The hydrogens and the Ca²⁺ ion are completed from the template:
the ion from the known heavy atoms within 3 Å of it in the MD structure.

The reference structure fixes the canonical pose of the features and the
pocket: the protein tokens within 6 Å of the ligand. The folder also records
the ligand's atom order as BioEmu2 0.1.0rc0 builds it; a later bioemu2 that
orders SMILES ligands differently is refused instead of silently scrambling
the map.

In [ ]:
ONE_LETTER = {"ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "ASH": "D", "CYS": "C", "CYX": "C",
              "GLN": "Q", "GLU": "E", "GLH": "E", "GLY": "G", "HIS": "H", "HSD": "H", "HSE": "H",
              "HSP": "H", "HID": "H", "HIE": "H", "HIP": "H", "ILE": "I", "LEU": "L", "LYS": "K",
              "LYN": "K", "MET": "M", "PHE": "F", "PRO": "P", "SER": "S", "THR": "T", "TRP": "W",
              "TYR": "Y", "VAL": "V"}
sequence = "".join(ONE_LETTER[res.resname] for res in protein.residues)
Path("bioemu2_system.yaml").write_text(
    "entities:\n"
    f"- entity_type: protein\n  asym_ids: [A]\n  sequence: {sequence}\n"
    f"- entity_type: ligand\n  asym_ids: [L]\n  smiles: {LIGAND_SMILES}\n")

result = subprocess.run(
    [sys.executable, "-m", "aimmd.cgaimmd.bioemu2", "build-system",
     "--bioemu2-system", "bioemu2_system.yaml", "--md-structure", "run.gro",
     "--topology", "topol.top", "--n-atoms", str(N_SOLUTE), "--out", "system",
     "--reference-structure", str(REFERENCE), "--pocket-cutoff", "6.0",
     "--msa", "query-only", "--device", "cuda"],
    capture_output=True, text=True)
print(result.stdout[-3000:], result.stderr[-2000:])

## 3. The params file

What differs from tutorial 5:

- **the calcium site check.** The validity check is a function that runs
  `TopologyValidity` on protein and ligand and then checks the ion: at least
  five protein oxygens within 3.0 Å of it, and no heavy atom closer than
  2.0 Å. `PolishedValidity` runs it on each raw sample and again after the
  polish of the whole solute, and copies its record (`info_keys`) into the
  sidecars;
- **the network** reads pooled features (ligand, pocket, whole protein) and
  adapts the pair embedding too: both are the defaults for a system with a
  ligand;
- **the fit** weights the ligand atoms five times in the denoising loss
  (without it the adapters learned to ignore the label) and trains the
  generator from the first round, once 200 frames exist, because plain BioEmu2
  puts most of its samples in the bound state.

In [ ]:
PARAMS = f'''
"""BioEmu2 C-GAIMMD on trypsin with benzamidine (tutorial 6)."""
import MDAnalysis as mda
import numpy as np

from aimmd.cgaimmd.bioemu2 import BioEmu2Network, sequential_fit
from aimmd.cgaimmd.preparation.descriptors import GeneratorAtoms, GeneratorDescriptors
from aimmd.cgaimmd.preparation.minimize import GromacsMinimizer
from aimmd.cgaimmd.preparation.solvate import GromacsSolvator, NVTStage
from aimmd.cgaimmd.preparation.topology import read_masses
from aimmd.cgaimmd.preparation.validity import PolishedValidity, TopologyValidity
from aimmd.cgaimmd.types import ValidityResult
from states import states_function

GMX = {GMX!r}
THREADS = {THREADS}
N_PROTEIN_LIGAND, N_SOLUTE = {N_PROTEIN_LIGAND}, {N_SOLUTE}
engine = 'gromacs'
chain_type = 'cgaimmd'
states = 'ARB'                                   # A bound, B unbound
topology = 'run.gro'
gmx_mdp = 'run.mdp'
gmx_grompp = f'{{GMX}} grompp -maxwarn 5 -p topol.top'
gmx_mdrun = f'{{GMX}} mdrun -ntmpi 1 -ntomp {{THREADS}}'
gmx_eneconv = ''
initial_paths = ['initial_A.xtc', 'initial_B.xtc']
free_seeding_position = 'random'
free_restart_source = 'equilibrium'
round_shooting_index = True
masses_from_topology = True
retry_with_state_definition_glitches = True
gen_temperature = 300.0
max_length = 50000
reweight_parameters = {{'free_threshold': 20}}
cgaimmd_parameters = {{'lambda_tolerance': 1.0, 'window_after_trained_shots': 1000,
                      'n_targets': 16, 'candidates_per_target': 2, 'generation_seed': 1}}

DESCRIPTORS = GeneratorDescriptors('run.gro', GeneratorAtoms(n_solute=N_SOLUTE),
                                   masses=read_masses('topol.top'))


def descriptors_function(trajectory):
    return DESCRIPTORS(trajectory)


class Network(BioEmu2Network):
    def __init__(self):
        super().__init__('system')


network = Network()

# the calcium site: protein oxygens around the ion, heavy atoms too close to it
_solute = mda.Universe('solute.gro')
_ION = N_SOLUTE - 1
_PROTEIN_OXYGENS = _solute.select_atoms('protein and name O*').indices
_HEAVY = np.setdiff1d(_solute.select_atoms('not name H*').indices, [_ION])


def calcium_site(x):
    """(ok, oxygens within 3.0 A, closest heavy atom in A) of every sample (n, N_SOLUTE, 3)."""
    d_oxygen = np.linalg.norm(x[:, _PROTEIN_OXYGENS] - x[:, _ION, None], axis=-1)
    d_heavy = np.linalg.norm(x[:, _HEAVY] - x[:, _ION, None], axis=-1)
    oxygens = (d_oxygen < 3.0).sum(axis=1)
    closest = d_heavy.min(axis=1)
    return (oxygens >= 5) & (closest >= 2.0), oxygens, closest


_protein_ligand = GeneratorDescriptors('protein_ligand.gro', GeneratorAtoms(n_solute=N_PROTEIN_LIGAND))
_topology_validity = TopologyValidity(
    'topol_protein_ligand.top', _protein_ligand, node_labels=np.arange(N_PROTEIN_LIGAND),
    stereocenters=[q for q in network.system.stereocenters() if max(q) < N_PROTEIN_LIGAND],
    clash_reference=['clash_reference_protein_ligand.xtc'])


def _checked(positions, context):
    """The topology check of protein and ligand, then the calcium site."""
    head = _topology_validity(positions[:, :N_PROTEIN_LIGAND], context)
    x = np.concatenate([np.asarray(head.positions, dtype=float),
                        np.asarray(positions[:, N_PROTEIN_LIGAND:], dtype=float)], axis=1)
    site_ok, oxygens, closest = calcium_site(x)
    ok = np.asarray(head.ok, dtype=bool) & site_ok
    reason = np.where(np.asarray(head.ok, dtype=bool),
                      np.where(site_ok, '', 'calcium_site'),
                      np.asarray(head.reason, dtype=object)).astype(object)
    info = [dict(ca_oxygens=int(n), ca_closest_heavy_A=float(d)) for n, d in zip(oxygens, closest)]
    return ValidityResult(positions=x, ok=ok, reason=reason, info=info)


_polisher = GromacsMinimizer(em_mdp='em.mdp', restrain='generator_heavy',
                             topology='topol_solute.top', grompp=f'{{GMX}} grompp -maxwarn 5',
                             mdrun=f'{{GMX}} mdrun -ntmpi 1 -ntomp 1 -nb cpu -pin off')
_validity = PolishedValidity(_checked, _polisher, 'solute.gro', workers=THREADS,
                             info_keys=('ca_oxygens', 'ca_closest_heavy_A'))


def cgaimmd_validity_function(positions, context):
    return _validity(positions, context)


_solvator = GromacsSolvator(DESCRIPTORS, topology='topol.top', em_mdp='em.mdp',
                            nvt_stages=[NVTStage('relax.mdp', 20.0, gen_vel=True)],
                            grompp=f'{{GMX}} grompp', mdrun=f'{{GMX}} mdrun -ntmpi 1 -ntomp {{THREADS}}',
                            maxwarn=5)


def cgaimmd_prepare_function(positions, context):
    return _solvator(positions, context)


_minimizer = GromacsMinimizer(em_mdp='em.mdp', restrain='generator_heavy', restraint_k=1e4,
                              emtol=100.0, nsteps=1000, grompp=f'{{GMX}} grompp',
                              mdrun=f'{{GMX}} mdrun -ntmpi 1 -ntomp {{THREADS}}', maxwarn=5)


def cgaimmd_minimize_function(frame, context):
    return _minimizer(frame, context)


def fit(params, pathensemble, verbose=False, worker=None):
    return sequential_fit(params, pathensemble, verbose=True, worker=worker,
                          min_shots_for_generator=0, min_frames_for_generator=200,
                          adapter_steps=3000, ligand_weight=5.0)
'''

for state in "AB":
    md_u = mda.Universe("run.gro", str(FREE_MD[state]))
    with mda.Writer(f"initial_{state}.xtc", md_u.atoms.n_atoms) as writer:
        for ts in md_u.trajectory[::200]:
            writer.write(md_u.atoms)

Path("params.py").write_text(PARAMS.lstrip())
params = aimmd.Params.load("params.py", save=False)
print(params.network.system.summary())
print("readout:", params.network.readout)

### How many unbiased frames pass the calcium check

The check must accept the unbiased MD, or it would reject good candidates.
Run it on the free-MD frames of both states before the campaign.

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("trypsin_params", "params.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
for state in "AB":
    md_u = mda.Universe("run.gro", str(FREE_MD[state]))
    x = module.DESCRIPTORS(md_u.trajectory[::100]).reshape(-1, N_SOLUTE, 3)   # whole molecules
    ok, oxygens, closest = module.calcium_site(x)
    print(f"state {state}: {ok.mean():.3f} of {len(ok)} frames pass the calcium site check")

## 4. The initial network, the preflight and the launch

As in tutorial 5: the committor head is pretrained on unbiased in-state frames,
balanced between the two states; the preflight runs on a GPU; and the job
script is written on the cluster.

In [ ]:
from aimmd.cgaimmd.bioemu2 import pretrain_initial_network
from aimmd.cgaimmd.flow_matching.pretrain import collect_end_state_frames

frames = collect_end_state_frames([str(FREE_MD["A"]), str(FREE_MD["B"])], structure="run.gro",
                                  descriptors_function=params.descriptors_function,
                                  states_function=states_function, states="ARB", stride=20,
                                  max_frames=20000)
codes = np.where(frames["state"] == "A", 1, 2)
record = pretrain_initial_network(params.network.to("cuda"), frames["coordinates"], codes,
                                  network_file="pretrained/networkARB.h5", verbose=True)
with open("params.py", "a") as handle:
    handle.write("\ninitial_network = 'pretrained/networkARB.h5'\n")
params = aimmd.Params.load("params.py", save=False)

result = subprocess.run([sys.executable, "-m", "aimmd.cgaimmd.preflight", "params.py",
                         "--candidates", "4", "--device", "cuda"], capture_output=True, text=True)
print(result.stdout)

In [ ]:
params.slurm_header = "#SBATCH --account=<project>\n#SBATCH --partition=<gpu partition>"
launcher = aimmd.Launcher(params, str(RUN))
launcher.create_job(
    "job.sh", 5, 1, 1, walltime=12 * 3600, ntasks_per_node=4, cpus_per_task=THREADS,
    gpus_per_task=1, skip_binding=False,
    preamble=["module load Stages/2026 nvidia-compilers/25.9-CUDA-13 OpenMPI/5.0.8 GCC/14.3.0",
              "module load GROMACS/2025.4-CUDA-13",
              "source /path/to/project/cgaimmd_venv/bin/activate",
              "export HF_HOME=/path/to/project/cache/huggingface HF_HUB_OFFLINE=1"],
    preflight=True)
print(Path("job.sh").read_text())

## 5. Monitoring the calcium site

The sidecars carry the calcium-site record of each shot's raw and polished
sample (`validity.info`), and the refill counts how many candidates failed the
site check (`calcium_site`).

In [ ]:
from aimmd.cgaimmd.sidecar import read_shot_sidecar, shot_table, summarise_shots

rows = shot_table(RUN, params=params)
summary = summarise_shots(rows, frame_interval_ps=20.0)
print("shots:", summary["n_shots"], "| reactive share:", summary["reactive_share"])
print("rejected in the refills:", summary["stage1_rejected"])
last = read_shot_sidecar(str(RUN / rows[-1]["chain"] / rows[-1]["path"]))
print(json.dumps(last["validity"], indent=1)[:1500])

## Where to go from here

- {doc}`../cgaimmd` describes the validity, preparation and minimization hooks
  and the sidecars in full.
- {doc}`../cgaimmd` also describes C-GAIMMD under a static bias, for example
  a frozen OPES bias on the bound state: what is reweighted, why candidate
  preparation runs without the bias, and the zero point.